In [ ]:
from bs4 import BeautifulSoup
import requests, os, csv, urllib.parse, urllib3, time, random
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry


from fuzzywuzzy import fuzz

import numpy as np
import pandas as pd

import string
import time
import re
urllib3.disable_warnings()

import os
import glob

from tqdm import tqdm

def preprocessing(sent):
    if isinstance(sent,str)==True and sent != "ABCXYZ":
        sent = re.sub(r"\([^()]*\)", "", sent ) #remove brackets and words within
        sent=sent.replace("&", "and") #change & to and
        sent=re.sub(r'[^\w\s]', '', sent) #remove all punct
        sent=sent.lower() #lower case everything
       
        #not working, nostra information technology pty ltd - need to test
        #Below code moves words presented in the list 
        list_word_remove=["nec","other","ltd","pty","cc"]
        wordlist=sent.split() ##Physiotherapy physiapy "other"
        sent=[x for x in wordlist if x not in list_word_remove]
        sent=' '.join(word for word in sent)
        sent=re.sub(r"\s+"," ", sent, flags = re.I) #remove mulit spaces
        
    return sent
    
retry_strategy = Retry(
    total=20,
    status_forcelist=[429, 500, 502, 503, 504],
    method_whitelist=["HEAD", "GET", "OPTIONS"],
    backoff_factor=2
)

adapter = HTTPAdapter(max_retries=retry_strategy)
http = requests.Session()
http.mount("https://", adapter)
http.mount("http://", adapter)

def get_proxies():
    proxyList = [
        {"http":"169.159.179.248:8080"}, {"http":"105.208.17.58:8080"}, {"http":"160.119.44.210:8080"}, {"http":"196.214.145.106:80"},
        {"https":"41.194.37.106:45381"},{"https":"41.222.159.191:8080"},{"https":"66.251.179.207:8080"}
        ]
    return proxyList

def rotate_proxy(proxies):
    proxy_select = -1
    if int(os.getenv('proxy_enabled', default=0)) > 0:
        proxy_select = random.randint(-1,len(proxies)-1)
    if proxy_select < 0:
        time.sleep(random.uniform(0.2,0.7))
        return None
    else:
        print(f"Proxy -> {proxies[proxy_select].get('http','')}{proxies[proxy_select].get('https','')}")
        return proxies[proxy_select]

def get_user_agents():
    userAgents = [
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/86.0.4240.193 Safari/537.36",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv: 82.0) Gecko/20100101 Firefox/82.0",
        "Mozilla/5.0 (Windows NT 10.0; Trident/7.0; rv:11.0) like Gecko",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/86.0.4240.193 Safari/537.36 Edg/86.0.622.63"
        ]
    return userAgents

def rotate_user_agent(userAgents):
    return userAgents[random.randint(0,len(userAgents)-1)]



def loadCompanyNames(fileLocation):
    lastCheckpoint = getLastCheckpoint()
    lastId = lastCheckpoint.get("id",-1)
    with open(fileLocation, mode='r', newline='\n', encoding='utf-8') as infile:
        csv_reader = csv.DictReader(infile, delimiter=',', quotechar='"')
        for row in csv_reader:
            if int(row[""]) > int(lastId):
                yield {"id":str(row[""]) , "file":f"{os.getenv('input_file')}","name":str(row['CUST_FULL_NAME_1'])}
            else:
                continue

def writeToCsv(result):
    with open(os.path.join("data", "output", f"out_{os.getenv('input_file')}"), mode='a+', newline='\n',
              encoding='utf-8') as infile:
        writer = csv.DictWriter(infile, result.keys())
        writer.writerow(result)

def updateCheckpoint(id):
    with open(os.path.join("data", "checkpoints", f"checkpoint_{os.getenv('input_file')}"), mode='w+', newline='\n',
              encoding='utf-8') as infile:
        writer = csv.DictWriter(infile, ["id","fileName"])
        writer.writeheader()
        writer.writerow({"id":f"{str(id)}", "fileName":f"{os.getenv('input_file')}"})

def getLastCheckpoint():
    try:
        with open(os.path.join("data", "checkpoints", f"checkpoint_{os.getenv('input_file')}"), mode='r', newline='\n',
                  encoding='utf-8') as infile:
            csv_reader = csv.DictReader(infile, delimiter=',', quotechar='"')
            for row in csv_reader:
                if row:
                    return row
                else:
                    break
            else:
                return {}
    except FileNotFoundError:
        return {}
    except FileExistsError:
        return {}

def buildSearchUrl(queryInput):
    return f"https://google.com/search?q={urllib.parse.quote(queryInput)}"


def makeRequest(url, header,proxy):
    return http.get(url, headers=header, proxies=proxy, verify=False)


def parse_listing(url, proxies):
    proxy = rotate_proxy(proxies)
    user_agent = rotate_user_agent(get_user_agents())
    headers = {'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
               'Accept-Language': 'en-US,en;q=0.9,af;q=0.8',
               'Accept-Encoding': 'gzip, deflate, br',
               'Connection': 'keep-alive',
               'Upgrade-Insecure-Requests': '1',
               'User-Agent': user_agent}
    try:
        req = makeRequest(url, headers, proxy)
    except:
        return None

    if req:
        soup = BeautifulSoup(req.content, 'lxml')
    else:
        return None
    if not soup:
        return None
    else:
        div = soup.find("div", class_="SPZz6b")
        name = None
        if div:
            for tag in div:
                if tag.attrs.get("data-attrid") == 'title':
                    name = tag.string
                    break
                else:
                    name = None
        else: name = None

        industry = soup.find("span", class_='YhemCb')
        if industry: industry = industry.text
        else: industry = None
            
        '''enter other code attributes here'''

        if name or industry:
            return {"name":name, "industry":industry}
        else:
            return None

def cleanName(name):
    strName = name.lower()
    strName = strName.replace("(", "").replace(")", "")
    strName = strName.replace("pty","").replace("ltd","").replace("(ltd)","")
    strName = strName.replace("pty ltd", "").replace("pty(ltd)", "")
    strName = strName.replace("cc", "").replace("cc.", "")
    strName = strName.replace("pty.", "").replace("ltd.", "")
    return strName.capitalize()

def cleanBusinessName(name):
    if "*" in name: return True, cleanName(name.replace("*", "").replace("/", "_"))
    else: return False, f"{name} skip individual"


def run():
    for i, record in enumerate(loadCompanyNames(os.path.join("data","input",f"{os.getenv('input_file')}"))):
        company, businessName = cleanBusinessName(record.get("name"))
        if company:
            outcome = parse_listing(buildSearchUrl(businessName), get_proxies())
            if outcome:
                result = {"file": os.getenv('input_file'), "id": str(record.get("id")),
                          "cis_name": record.get("name"), "search_name": businessName,
                          "matching_name": outcome.get("name"), "industry": outcome.get("industry")
                          }
                print(f"{str(record.get('id'))} > {record.get('name')} > {outcome.get('name')} - {outcome.get('industry')}")
            else:
                result = {"file": os.getenv('input_file'), "id": str(record.get("id")),
                          "cis_name": record.get("name"), "search_name": businessName,
                          "matching_name": None, "industry": None
                          }
                print(f"{str(record.get('id'))} > {businessName} no match")
            writeToCsv(result)
        else:
            print(f"{str(record.get('id'))} > {businessName}")

        updateCheckpoint(record.get("id"))

        
df=pd.read_csv(r"webscrape.csv")
#df["CUST_NAME"]=df["search"]
df["is_class"]="No"
df["Google_name"]=''
df["Google_indus"]=''
df["Fuzzy_match"]=''
df["tried"]="No"

col_names=["CUST_NAME","CUST_NO","is_class","Google_name","Google_indus","Fuzzy_match","tried"]
df=df[col_names]
print("Original file: ",df.shape[0])

path = 'Save_new\\'              # use your path
all_files = glob.glob(os.path.join(path, "*.csv")) 

lis=[]
for file in tqdm(all_files):
    if file.endswith(".csv"):  
        df_=pd.read_csv(file)
        lis.append(df_)
df_master = pd.concat(lis, axis=0, ignore_index=True)
df_master=df_master[col_names]

print("webscraped: ",df_master.shape[0])

df=pd.concat([df,df_master])
df=df.drop_duplicates(subset=["CUST_NO"],keep=False)
print("Left: ",df.shape[0])

os.chdir( 'Save_new\\')
start=time.time()

for row in tqdm(range(df.shape[0])):

    #print ("Busy with row",row)
    
    name=df.iloc[row,df.columns.get_loc("CUST_NAME")]
    num=df.iloc[row,df.columns.get_loc("CUST_NO")]
    
    pop_data=[[name,num,"","","","","Yes"]]
    if re.search(r'\bta\b',str(name).lower()) is None:
        outcome = parse_listing(buildSearchUrl(name), get_proxies())
        if outcome is not None:
            if outcome.get("name", 0):
                if outcome.get("industry", 0):
                    google_name=outcome.get("name", 0)
                    google_indus=outcome.get("industry", 0)
                    Token_Sort_Ratio = fuzz.token_sort_ratio(name,google_name)       
                    pop_data=[[name,num,"Yes",google_name,google_indus,Token_Sort_Ratio/100,"Yes"]]
                    
    hold_df=pd.DataFrame(data=pop_data,columns=col_names)  
    hold_df.to_csv(str(num)+".csv")
    #print (end="\r")
print(time.time()-start)

C:\Users\F5448255\AppData\Local\Continuum\anaconda3\lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


Original file:  200000


100%|█████████████████████████████████████████████████████████████████████████| 185423/185423 [07:48<00:00, 396.16it/s]


webscraped:  185423
Left:  14579


 20%|██████████████▋                                                         | 2984/14579 [5:06:51<19:59:31,  6.21s/it]

In [3]:
import pandas as pd
import os
import glob
import re
from tqdm import tqdm



path = r'C:\Users\F5448255\OneDrive - FRG\Documents\Web scrapping\Save_new' # use your path
all_files = glob.glob(os.path.join(path, "*.csv"))
lis=[]
for file in tqdm(all_files):
    if file.endswith(".csv"):
   

        df=pd.read_csv(file)
        lis.append(df)
df_1= pd.concat(lis, axis=0, ignore_index=True)
df_1.head(3)



df_1.to_csv("save_H.csv")

100%|█████████████████████████████████████████████████████████████████████████| 188428/188428 [22:11<00:00, 141.50it/s]


In [ ]:
from tqdm import tqdm
for row in tqdm(range(10)):
    print(row)